In [1]:
#session
import os, re
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

BASE = os.path.expanduser("~/bosch-bigdata")
P = f"{BASE}/data/parquet"

spark = (SparkSession.builder
         .appName("BoschPreparation")
         .master("local[*]")
         .config("spark.driver.memory", "5g")
         .config("spark.sql.shuffle.partitions", "200")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

num = spark.read.parquet(f"{P}/train_numeric")
cat = spark.read.parquet(f"{P}/train_categorical")
dat = spark.read.parquet(f"{P}/train_date")
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 15:58:05 WARN Utils: Your hostname, Vamoussa, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/14 15:58:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 15:58:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

4.2.0


In [2]:
#2.6 Nettoyage:variables numériques
manquants = pd.read_csv(f"{BASE}/export/manquants_numeric.csv", index_col=0).iloc[:, 0]
seuil = 0.99
quasi_vides = manquants[manquants > seuil].index.tolist()
candidates = manquants[manquants <= seuil].index.tolist()

distincts = num.select([F.approx_count_distinct(c).alias(c) for c in candidates]).toPandas().iloc[0]
constantes = distincts[distincts <= 1].index.tolist()
a_garder = [c for c in candidates if c not in constantes]

print(f"Colonnes de départ          : {len(manquants)}")
print(f"Retirées (> 99 % manquants) : {len(quasi_vides)}")
print(f"Retirées (constantes)       : {len(constantes)}")
print(f"Colonnes conservées         : {len(a_garder)}")

num.select("Id", *a_garder, "Response").write.mode("overwrite").parquet(f"{P}/num_propre")
pd.Series(quasi_vides + constantes, name="colonne_retiree").to_csv(
    f"{BASE}/export/colonnes_retirees_numeric.csv", index=False)

/home/diaba/bosch-env/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
                                                                                

Colonnes de départ          : 968
Retirées (> 99 % manquants) : 227
Retirées (constantes)       : 0
Colonnes conservées         : 741


In [4]:
#variables catégorielles
cols_cat = [c for c in cat.columns if c != "Id"]
total = cat.count()

remplissage = cat.select([(F.count(c) / total).alias(c) for c in cols_cat]).toPandas().iloc[0]
cat_candidates = remplissage[remplissage >= 0.01].index.tolist()

dist_cat = cat.select([F.approx_count_distinct(c).alias(c) for c in cat_candidates]).toPandas().iloc[0]
cat_garder = dist_cat[dist_cat >= 2].index.tolist()

print(f"Colonnes catégorielles          : {len(cols_cat)}")
print(f"Remplies pour au moins 1 %      : {len(cat_candidates)}")
print(f"Conservées (au moins 2 valeurs) : {len(cat_garder)}")

/home/diaba/bosch-env/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/home/diaba/bosch-env/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
[Stage 24:>                                                         (0 + 1) / 1]

Colonnes catégorielles          : 2140
Remplies pour au moins 1 %      : 313
Conservées (au moins 2 valeurs) : 14


In [5]:
#2.7 Ingénierie des variables: variables de parcours (2.7.1)
parcours = spark.read.parquet(f"{P}/parcours").drop("Response")
stations = sorted(int(c.split("_S")[1]) for c in parcours.columns if c.startswith("visite_S"))

ligne_de = {}
for c in dat.columns:
    m = re.match(r"L(\d)_S(\d+)_D\d+", c)
    if m:
        ligne_de[int(m.group(2))] = int(m.group(1))

v = lambda s: F.col(f"visite_S{s}")
feat_parcours = (parcours
    .withColumn("nb_stations", sum(v(s) for s in stations))
    .withColumn("premiere_station", F.least(*[F.when(v(s) == 1, F.lit(s)) for s in stations]))
    .withColumn("derniere_station", F.greatest(*[F.when(v(s) == 1, F.lit(s)) for s in stations])))

for l in range(4):
    st_l = [s for s in stations if ligne_de[s] == l]
    col_l = F.greatest(*[v(s) for s in st_l]) if len(st_l) > 1 else v(st_l[0])
    feat_parcours = feat_parcours.withColumn(f"ligne_L{l}", col_l)

feat_parcours.write.mode("overwrite").parquet(f"{P}/feat_parcours")
print(len(feat_parcours.columns) - 1, "variables de parcours")

[Stage 26:====================>                                    (4 + 7) / 11]

59 variables de parcours


In [6]:
#variables temporelles (2.7.2)
temps = spark.read.parquet(f"{P}/temps").drop("Response")
w = Window.orderBy("t_debut", "Id")

feat_temps = (temps
    .withColumn("ecart_prec", F.col("t_debut") - F.lag("t_debut").over(w))
    .withColumn("ecart_suiv", F.lead("t_debut").over(w) - F.col("t_debut"))
    .withColumn("diff_id_prec", F.col("Id") - F.lag("Id").over(w))
    .withColumn("diff_id_suiv", F.lead("Id").over(w) - F.col("Id"))
    .withColumn("nb_meme_debut", F.count("*").over(Window.partitionBy("t_debut"))))

feat_temps.write.mode("overwrite").parquet(f"{P}/feat_temps")
print(len(feat_temps.columns) - 1, "variables temporelles")

[Stage 30:>                                                         (0 + 1) / 1]

9 variables temporelles


In [7]:
#résumé numérique et encodage des catégories (2.7.3)
from pyspark.ml.feature import StringIndexer

num_propre = spark.read.parquet(f"{P}/num_propre")
cols_num = [c for c in num_propre.columns if c not in ("Id", "Response")]
m = F.filter(F.array(*cols_num), lambda x: x.isNotNull())

feat_resume = num_propre.select("Id",
    F.size(m).alias("nb_mesures"),
    F.array_min(m).alias("mesure_min"),
    F.array_max(m).alias("mesure_max"),
    F.when(F.size(m) > 0,
           F.aggregate(m, F.lit(0.0), lambda acc, x: acc + x) / F.size(m)).alias("mesure_moy"))
feat_resume.write.mode("overwrite").parquet(f"{P}/feat_resume")

cat_sel = cat.select("Id", *cat_garder)
indexeur = StringIndexer(inputCols=cat_garder,
                         outputCols=[f"{c}_idx" for c in cat_garder],
                         handleInvalid="keep")
cat_idx = indexeur.fit(cat_sel).transform(cat_sel).select("Id", *[f"{c}_idx" for c in cat_garder])
cat_idx.write.mode("overwrite").parquet(f"{P}/cat_idx")
print("Résumé numérique et catégories encodées enregistrés")

[Stage 39:>                                                       (0 + 10) / 10]

Résumé numérique et catégories encodées enregistrés


In [8]:
#2.8 Jeu final : jointure, valeurs manquantes et découpage (la plus longue, plusieurs minutes)
final = (num_propre
         .join(spark.read.parquet(f"{P}/feat_parcours"), "Id")
         .join(spark.read.parquet(f"{P}/feat_temps"), "Id")
         .join(spark.read.parquet(f"{P}/feat_resume"), "Id")
         .join(spark.read.parquet(f"{P}/cat_idx"), "Id")
         .fillna(-999))

final = (final.withColumn("u", F.rand(seed=42))
              .withColumn("split", F.when(F.col("u") < 0.70, "train")
                                    .when(F.col("u") < 0.85, "valid")
                                    .otherwise("test"))
              .drop("u"))

final.write.mode("overwrite").partitionBy("split").parquet(f"{P}/final")
print("Jeu final enregistré")

Jeu final enregistré


In [9]:
#vérification (Tableau 2.5)
final = spark.read.parquet(f"{P}/final")
(final.groupBy("split")
      .agg(F.count("*").alias("pieces"), F.sum("Response").alias("defauts"))
      .withColumn("taux_%", F.round(100 * F.col("defauts") / F.col("pieces"), 3))
      .orderBy(F.desc("pieces")).show())

print("Nombre de variables explicatives :", len(final.columns) - 3)
!du -sh {P}/final

+-----+------+-------+------+
|split|pieces|defauts|taux_%|
+-----+------+-------+------+
|train|828978|   4839| 0.584|
|valid|177451|   1027| 0.579|
| test|177318|   1013| 0.571|
+-----+------+-------+------+

Nombre de variables explicatives : 827
391M	/home/diaba/bosch-bigdata/data/parquet/final
